<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/Plotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import plotly.express as px
import kagglehub as ka

path = ka.dataset_download("mustafabayar/europe-sales-records")
df = pd.read_csv(f"{path}/Europe Sales Records.csv")



100%|██████████| 49.9k/49.9k [00:00<00:00, 21.9MB/s]

Extracting files...


In [2]:
df


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Europe,Czech Republic,Beverages,Offline,C,9/12/2011,478051030,9/29/2011,4778,47.45,31.79,226716.10,151892.62,74823.48
1,Europe,Bosnia and Herzegovina,Clothes,Online,M,10/14/2013,919133651,11/4/2013,927,109.28,35.84,101302.56,33223.68,68078.88
2,Europe,Austria,Cereal,Offline,C,8/13/2014,987410676,9/6/2014,5616,205.70,117.11,1155211.20,657689.76,497521.44
3,Europe,Bulgaria,Office Supplies,Online,L,10/31/2010,672330081,11/29/2010,6266,651.21,524.96,4080481.86,3289399.36,791082.50
4,Europe,Estonia,Fruits,Online,L,9/28/2016,579463422,11/1/2016,4958,9.33,6.92,46258.14,34309.36,11948.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1325,Europe,Norway,Personal Care,Offline,M,1/14/2014,634033286,1/15/2014,3394,81.73,56.67,277391.62,192337.98,85053.64
1326,Europe,Ukraine,Cereal,Offline,L,4/14/2014,559183347,5/21/2014,3633,205.70,117.11,747308.10,425460.63,321847.47
1327,Europe,Armenia,Meat,Offline,M,11/9/2015,781416594,12/23/2015,7390,421.89,364.69,3117767.10,2695059.10,422708.00
1328,Europe,Denmark,Clothes,Offline,H,5/9/2012,713357150,6/3/2012,7088,109.28,35.84,774576.64,254033.92,520542.72


### Galería de Gráficos de Plotly Express con tus Datos

A continuación, encontrarás ejemplos prácticos de los principales tipos de gráficos que ofrece `plotly.express` adaptados a este DataFrame.

In [3]:
# Asegurar que las fechas estén en formato datetime si no se han procesado antes
if df['Order Date'].dtype == 'object':
    df['Order Date'] = pd.to_datetime(df['Order Date'])
    df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Preparar datos agregados para algunos gráficos
df_monthly = df.groupby(df['Order Date'].dt.to_period('M'))['Total Profit'].sum().reset_index()
df_monthly['Order Date'] = df_monthly['Order Date'].dt.to_timestamp()

profit_by_item = df.groupby('Item Type')['Total Profit'].sum().reset_index().sort_values(by='Total Profit', ascending=True)

### 1. Gráfico de Líneas (Line Chart)
**¿Para qué sirve?**
Este gráfico se utiliza principalmente para mostrar tendencias y cambios en los datos a lo largo de intervalos de tiempo continuos (análisis de series temporales).

**¿Qué información ofrece aquí?**
Muestra la evolución temporal de las ganancias mensuales totales (`Total Profit`), permitiendo identificar patrones de estacionalidad, crecimiento general o caídas de rendimiento a lo largo de los meses.

In [13]:
# 1. Gráfico de Líneas (Line Chart)
# ¿Para qué sirve?: Muestra tendencias y variaciones a lo largo del tiempo (series temporales).
# ¿Qué información ofrece?: Representa la evolución mensual del beneficio neto ('Total Profit') para identificar meses de alta o baja rentabilidad.
fig_line = px.line(
    df_monthly,
    x='Order Date',
    y='Total Profit',
    title='Línea: Evolución Temporal de Ganancias Mensuales',
    markers=True
)
fig_line.show()

### 2. Gráfico de Barras (Bar Chart)
**¿Para qué sirve?**
Es ideal para comparar cantidades o magnitudes numéricas entre diferentes categorías discretas.

**¿Qué información ofrece aquí?**
Compara el beneficio total generado por cada tipo de producto (`Item Type`). El color añade una dimensión visual adicional para identificar rápidamente cuáles categorías de productos son las más y menos rentables de un solo vistazo.

In [14]:
# 2. Gráfico de Barras (Bar Chart)
# ¿Para qué sirve?: Compara cantidades o magnitudes numéricas entre categorías discretas.
# ¿Qué información ofrece?: Compara el beneficio acumulado por tipo de producto ('Item Type'), ordenado para resaltar rápidamente los más rentables.
fig_bar = px.bar(
    profit_by_item,
    x='Item Type',
    y='Total Profit',
    color='Total Profit',
    title='Barras: Ganancia Total por Tipo de Producto'
)
fig_bar.show()

### 3. Gráfico de Dispersión (Scatter Plot)
**¿Para qué sirve?**
Permite analizar la relación, dispersión o correlación real entre dos variables numéricas que no tienen una dependencia lineal estricta u obvia, ayudando a descubrir patrones complejos.

**¿Qué información ofrece aquí?**
Muestra las **Unidades Vendidas (`Units Sold`)** frente al **Beneficio Total (`Total Profit`)** de cada pedido, agrupado por **Tipo de Producto (`Item Type`)**. Esto permite ver claramente qué productos generan más margen con menor volumen de ventas (pendientes más inclinadas hacia arriba) y cuáles requieren grandes volúmenes para ser significativos.

In [15]:
# 3. Gráfico de Dispersión (Scatter Plot)
# ¿Para qué sirve?: Analiza la correlación y dispersión real entre dos variables cuantitativas sin dependencias matemáticas triviales.
# ¿Qué información ofrece?: Relaciona el volumen de ventas ('Units Sold') frente al beneficio real generado ('Total Profit'), agrupado por producto para descubrir el margen real por pedido.
fig_scatter = px.scatter(
    df,
    x='Units Sold',
    y='Total Profit',
    color='Item Type',           # Diferenciación por tipo de producto
    hover_data=['Country', 'Sales Channel'],
    title='Dispersión: Unidades Vendidas frente al Beneficio Total por Producto'
)
fig_scatter.show()

### 4. Gráfico de Sectores / Dona (Pie/Donut Chart)
**¿Para qué sirve?**
Muestra la distribución proporcional o la participación porcentual que tiene cada categoría con respecto al total general (las partes de un todo).

**¿Qué información ofrece aquí?**
Representa el porcentaje que aporta cada canal de venta (`Sales Channel`) al total de ingresos generados (`Total Revenue`). El agujero en el centro (diseño de dona) mejora la legibilidad en comparación con un gráfico de pastel tradicional.

In [16]:
# 4. Gráfico de Sectores / Dona (Pie/Donut Chart)
# ¿Para qué sirve?: Muestra la proporción o porcentaje que aporta cada categoría respecto a un total.
# ¿Qué información ofrece?: Indica la participación porcentual de los canales Online y Offline en los ingresos totales ('Total Revenue').
fig_pie = px.pie(
    df,
    names='Sales Channel',
    values='Total Revenue',
    hole=0.4,
    title='Dona: Proporción de Ingresos Totales por Canal de Venta'
)
fig_pie.show()

### 5. Histograma (Histogram)
**¿Para qué sirve?**
Se utiliza para representar la distribución de una variable numérica continua, agrupando los valores en rangos o intervalos específicos (bins) para ver su frecuencia.

**¿Qué información ofrece aquí?**
Muestra la distribución de la cantidad de unidades vendidas por pedido, revelando los rangos de volumen más frecuentes y cómo difieren estas distribuciones entre canales de venta online y offline.

In [17]:
# 5. Histograma (Histogram)
# ¿Para qué sirve?: Muestra la distribución y frecuencia de una variable continua dividiéndola en intervalos (bins).
# ¿Qué información ofrece?: Permite ver el tamaño de pedido más frecuente ('Units Sold') y comparar si el comportamiento difiere entre Online y Offline.
fig_hist = px.histogram(
    df,
    x='Units Sold',
    nbins=30,
    color='Sales Channel',
    title='Histograma: Distribución de Unidades Vendidas por Pedido'
)
fig_hist.show()

### 6. Gráfico de Caja (Box Plot)
**¿Para qué sirve?**
Representa de forma compacta la distribución, dispersión, simetría y los valores atípicos (outliers) de un conjunto de datos a través de sus cuartiles.

**¿Qué información ofrece aquí?**
Compara la distribución de los beneficios totales según el nivel de prioridad del pedido (`Order Priority`), permitiendo identificar visualmente la mediana del beneficio y la variabilidad de cada categoría.

In [18]:
# 6. Gráfico de Caja (Box Plot)
# ¿Para qué sirve?: Representa gráficamente la dispersión, cuartiles y los valores atípicos (outliers) de una distribución.
# ¿Qué información ofrece?: Compara los rangos de beneficio ('Total Profit') obtenidos según la prioridad asignada al pedido ('Order Priority').
fig_box = px.box(
    df,
    x='Order Priority',
    y='Total Profit',
    color='Order Priority',
    title='Caja: Distribución del Beneficio según la Prioridad del Pedido'
)
fig_box.show()

### 7. Gráfico de Violín (Violin Plot)
**¿Para qué sirve?**
Combina las características de un gráfico de caja con un gráfico de densidad de kernel. Es la herramienta perfecta para observar detalladamente la forma exacta de la distribución de los datos.

**¿Qué información ofrece aquí?**
Muestra la densidad y los rangos más comunes del costo unitario (`Unit Cost`) para cada canal de venta. Al incluir la caja interna, es sumamente fácil comparar los valores medianos y cuartiles al mismo tiempo que la densidad.

In [19]:
# 7. Gráfico de Violín (Violin Plot)
# ¿Para qué sirve?: Muestra la densidad de los datos a lo largo de su distribución junto con un resumen interno de caja.
# ¿Qué información ofrece?: Detalla y compara la distribución de los costos unitarios ('Unit Cost') entre las modalidades de venta Online y Offline.
import plotly.express as px

fig_violin = px.violin(
    df,
    y='Unit Cost',
    x='Sales Channel',
    color='Sales Channel',
    box=True,          # Muestra la caja interna con la mediana y cuartiles
    points='outliers', # Muestra solo los valores atípicos para no saturar el diseño
    title='Violín: Distribución del Costo Unitario por Canal de Venta'
)
fig_violin.show()

### 8. Treemap (Treemap Chart)
**¿Para qué sirve?**
Permite visualizar datos estructurados jerárquicamente utilizando rectángulos anidados, donde el tamaño y el color representan variables cuantitativas.

**¿Qué información ofrece aquí?**
Representa la jerarquía de las ganancias totales de cada tipo de producto. Las categorías con mayor proporción de ganancias ocupan un área rectangular más grande, complementado por una escala de color que resalta de verde (mayor beneficio) a rojo (menor beneficio).

In [20]:
# 8. Treemap (Treemap Chart)
# ¿Para qué sirve?: Representa datos jerárquicos o anidados mediante rectángulos proporcionales a su valor.
# ¿Qué información ofrece?: Proporciona una vista jerárquica del volumen de ganancia ('Total Profit') por tipo de artículo, codificado con colores de verde (mayor ganancia) a rojo.
fig_tree = px.treemap(
    df,
    path=['Item Type'],
    values='Total Profit',
    color='Total Profit',
    color_continuous_scale='RdYlGn',
    title='Treemap: Jerarquía de Ganancias por Categoría de Producto'
)
fig_tree.show()